# Müllklassifikation – Gesamtnotebook

Dieses Notebook vereint die komplette Pipeline des Projekts in **einer Datei**:

1. **Datenaufbereitung**
2. **Training**
3. **Vorhersage (Predictor)**

Dabei wurden die Inhalte der bisherigen Einzeldateien zusammengeführt, aber **nicht kaputt vereinfacht**.  
Wichtige Beschreibungen in Markdown bleiben erhalten oder wurden nur sprachlich gestrafft.


## Einleitung

Im Rahmen des Data-Lab-Kurses wurde ein Machine-Learning-Projekt entwickelt, das den praktischen Nutzen von Bildklassifikation im Kontext der Mülltrennung untersucht. Ziel des Projekts ist es, anhand von Bilddaten automatisiert zu erkennen, zu welcher Abfallkategorie ein Objekt gehört.

Damit adressiert das Projekt ein reales Problem: Fehlwürfe im Abfall führen zu hohen Sortierkosten, mindern die Recyclingqualität und erschweren ressourceneffizientes Handeln. Ein KI-gestütztes System könnte hier einen Beitrag leisten, indem es Nutzerinnen und Nutzer im Alltag unterstützt oder Sortierprozesse teilautomatisiert.

Ein zentraler Bestandteil des Projekts ist eine saubere und reproduzierbare Datenvorbereitung, die die Grundlage für jedes verlässliche Klassifikationsmodell bildet. Dieser Prozess umfasst die Sichtung, Analyse und Strukturierung des Datensatzes, das Vereinheitlichen der Bildgrößen, das Erstellen eines Train-Validation-Test-Splits sowie das Generieren künstlich erweiterter Trainingsdaten durch Datenaugmentation.


## Anleitung

1. **Run all**
2. Bei der Reset-Abfrage optional **`j`** eingeben, wenn `dataset-224`, `dataset-split` und `dataset-augmented` neu erzeugt werden sollen
3. Warten, bis das Notebook vollständig durchgelaufen ist
4. Danach kann unten im Predictor ein Bildpfad getestet werden


## 1. Setup & Bibliotheken

In [ ]:
import os
import stat
import shutil
import random
from collections import Counter

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf

from PIL import Image, ImageEnhance
from tensorflow import keras
from tensorflow.keras import layers, Sequential
from tensorflow.keras.preprocessing import image
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix


## 2. Projektpfad setzen und Datensatz prüfen

Wir setzen das Arbeitsverzeichnis auf unseren Projektordner und prüfen, ob die nötigen Ordner gefunden werden.


In [ ]:
DATASET_ORIGINAL_DIR = "dataset-original"

def find_project_root(start_path, required_names=None):
    if required_names is None:
        required_names = [DATASET_ORIGINAL_DIR]

    current = start_path
    while True:
        entries = set(os.listdir(current))
        if all(name in entries for name in required_names):
            return current

        parent = os.path.dirname(current)
        if parent == current:
            return None
        current = parent

print("Aktueller Arbeitsordner:", os.getcwd())

project_root = find_project_root(os.getcwd(), required_names=[DATASET_ORIGINAL_DIR])

if project_root is None:
    raise FileNotFoundError(
        f"'{DATASET_ORIGINAL_DIR}' konnte nicht gefunden werden. "
        "Starte das Notebook im Projektordner oder in einem Unterordner davon."
    )

if os.getcwd() != project_root:
    os.chdir(project_root)
    print("Arbeitsverzeichnis automatisch gesetzt auf:", project_root)
else:
    print("Arbeitsverzeichnis war schon korrekt gesetzt.")

print("dataset-original gefunden:", os.path.isdir(DATASET_ORIGINAL_DIR))
print("Inhalt des Projektordners:", os.listdir())


## 3. Alte generierte Ordner optional löschen

Diese Zelle löscht bei Bedarf alte generierte Ordner, damit Resize, Split und Augmentation sauber neu erzeugt werden können.

**Wichtig:** Erst den Projektpfad korrekt setzen, dann löschen.


In [ ]:
def handle_remove_readonly(func, path, exc):
    os.chmod(path, stat.S_IWRITE)
    func(path)

folders_to_reset = ["dataset-224", "dataset-split", "dataset-augmented"]

user_input = input("Ordner neu erzeugen? (j/n): ")

if user_input.lower() == "j":
    for folder in folders_to_reset:
        if os.path.exists(folder):
            shutil.rmtree(folder, onerror=handle_remove_readonly)
            print(f"Gelöscht: {folder}")
        else:
            print(f"Nicht vorhanden (ok): {folder}")
else:
    print("Reset übersprungen.")


## 4. Klassen automatisch einlesen

Wir lesen die vorhandenen Klassen automatisch aus `dataset-original` ein.  
Dadurch bleibt der Workflow flexibel, falls neue Kategorien hinzukommen.


In [ ]:
classes = sorted([
    d for d in os.listdir(DATASET_ORIGINAL_DIR)
    if os.path.isdir(os.path.join(DATASET_ORIGINAL_DIR, d))
])

print("Gefundene Klassen in dataset-original:", classes)


## 5. Beispielbilder anzeigen

Wir zeigen je Klasse einige Beispielbilder, um die Datenqualität zu prüfen.


In [ ]:
def show_examples(cls, n=4):
    folder = os.path.join(DATASET_ORIGINAL_DIR, cls)
    files = [f for f in os.listdir(folder) if os.path.isfile(os.path.join(folder, f))]
    n = min(n, len(files))

    if n == 0:
        print(f"Keine Bilder in Klasse: {cls}")
        return

    sample_files = random.sample(files, n)

    plt.figure(figsize=(10, 3))
    for i, file in enumerate(sample_files):
        img_path = os.path.join(folder, file)
        with Image.open(img_path) as img:
            img = img.copy()

        plt.subplot(1, n, i + 1)
        plt.imshow(img)
        plt.axis("off")

    plt.suptitle(f"Beispiele für: {cls}", y=1.02, fontsize=12)
    plt.tight_layout()
    plt.show()

for cls in classes:
    show_examples(cls, n=4)


## 6. Bildgrößen analysieren

Wir analysieren die Bildgrößen und die Häufigkeit der Bildgrößen. Das ist wichtig, bevor wir sie auf eine einheitliche Zielgröße von **224×224** skalieren.


In [ ]:
sizes = []

for cls in classes:
    folder = os.path.join(DATASET_ORIGINAL_DIR, cls)
    for file in os.listdir(folder):
        path = os.path.join(folder, file)
        if os.path.isfile(path):
            with Image.open(path) as img:
                sizes.append(img.size)

print("Häufigste Bildgrößen:", Counter(sizes).most_common(10))


In [ ]:
widths = [w for (w, h) in sizes]
heights = [h for (w, h) in sizes]

plt.figure(figsize=(6, 5))
plt.scatter(widths, heights, alpha=0.3)
plt.xlabel("Breite")
plt.ylabel("Höhe")
plt.title("Bildgrößen im Datensatz")
plt.show()


## 7. Resizing auf 224×224

Wir skalieren alle Bilder auf eine einheitliche Größe von **224×224 Pixeln** und speichern sie im Ordner `dataset-224`.


In [ ]:
INPUT_DIR = "dataset-original"
OUTPUT_DIR = "dataset-224"
TARGET_SIZE = (224, 224)

os.makedirs(OUTPUT_DIR, exist_ok=True)

for cls in classes:
    input_class = os.path.join(INPUT_DIR, cls)
    output_class = os.path.join(OUTPUT_DIR, cls)
    os.makedirs(output_class, exist_ok=True)

    for file in os.listdir(input_class):
        input_path = os.path.join(input_class, file)
        output_path = os.path.join(output_class, file)

        if not os.path.isfile(input_path):
            continue

        with Image.open(input_path) as img:
            img = img.convert("RGB")
            img = img.resize(TARGET_SIZE)
            img.save(output_path)

print("Resize abgeschlossen. Alle Bilder wurden nach dataset-224 resized.")


## 8. Train/Validation/Test Split

Wir splitten den Datensatz in **70% Training**, **15% Validation** und **15% Test**.


In [ ]:
INPUT_DIR = "dataset-224"
OUTPUT_BASE = "dataset-split"

for split in ["train", "val", "test"]:
    for cls in classes:
        os.makedirs(os.path.join(OUTPUT_BASE, split, cls), exist_ok=True)

train_ratio = 0.70
val_ratio = 0.15
test_ratio = 0.15

for cls in classes:
    files = [
        f for f in os.listdir(os.path.join(INPUT_DIR, cls))
        if os.path.isfile(os.path.join(INPUT_DIR, cls, f))
    ]

    train_files, temp_files = train_test_split(
        files,
        test_size=(1 - train_ratio),
        random_state=42
    )
    val_files, test_files = train_test_split(
        temp_files,
        test_size=(test_ratio / (test_ratio + val_ratio)),
        random_state=42
    )

    for f in train_files:
        shutil.copy(
            os.path.join(INPUT_DIR, cls, f),
            os.path.join(OUTPUT_BASE, "train", cls, f)
        )
    for f in val_files:
        shutil.copy(
            os.path.join(INPUT_DIR, cls, f),
            os.path.join(OUTPUT_BASE, "val", cls, f)
        )
    for f in test_files:
        shutil.copy(
            os.path.join(INPUT_DIR, cls, f),
            os.path.join(OUTPUT_BASE, "test", cls, f)
        )

print("Split abgeschlossen!")


In [ ]:
for split in ["train", "val", "test"]:
    print(split)
    for cls in classes:
        count = len(os.listdir(os.path.join("dataset-split", split, cls)))
        print(f"  {cls}: {count} Bilder")


In [ ]:
splits = ["train", "val", "test"]
fig, axes = plt.subplots(1, 3, figsize=(15, 4), sharey=True)

x = np.arange(len(classes))

for i, split in enumerate(splits):
    counts = [len(os.listdir(os.path.join("dataset-split", split, cls))) for cls in classes]
    axes[i].bar(x, counts)
    axes[i].set_title(f"{split.capitalize()} Split")
    axes[i].set_ylabel("Anzahl Bilder")
    axes[i].set_xticks(x)
    axes[i].set_xticklabels(classes, rotation=45)

plt.tight_layout()
plt.show()


In [ ]:
split_counts = {
    split: [len(os.listdir(os.path.join("dataset-split", split, cls))) for cls in classes]
    for split in splits
}

x = np.arange(len(classes))
width = 0.25

plt.figure(figsize=(12, 6))
for i, split in enumerate(splits):
    plt.bar(x + i * width, split_counts[split], width=width, label=split.capitalize())

plt.xticks(x + width, classes, rotation=45)
plt.ylabel("Anzahl Bilder")
plt.title("Train/Validation/Test – Datenverteilung je Klasse")
plt.legend()
plt.tight_layout()
plt.show()


In [ ]:
train_counts = [len(os.listdir(os.path.join("dataset-split/train", cls))) for cls in classes]
val_counts = [len(os.listdir(os.path.join("dataset-split/val", cls))) for cls in classes]
test_counts = [len(os.listdir(os.path.join("dataset-split/test", cls))) for cls in classes]

x = np.arange(len(classes))

plt.figure(figsize=(12, 6))
plt.bar(x, train_counts, label="Train")
plt.bar(x, val_counts, bottom=train_counts, label="Val")
plt.bar(x, test_counts, bottom=np.array(train_counts) + np.array(val_counts), label="Test")

plt.xticks(x, classes, rotation=45)
plt.ylabel("Anzahl Bilder")
plt.title("Gestapeltes Balkendiagramm – Datenverteilung je Klasse")
plt.legend()
plt.tight_layout()
plt.show()


## 9. Datenaugmentation

Wir erweitern die Trainingsdaten um Rotation, Spiegelung sowie Helligkeits- und Kontrastvariationen.


In [ ]:
INPUT_DIR = "dataset-split/train"
OUTPUT_DIR = "dataset-augmented/train"

os.makedirs(OUTPUT_DIR, exist_ok=True)

def augment_image(img):
    if random.random() < 0.5:
        img = img.rotate(random.randint(-15, 15))
    if random.random() < 0.5:
        img = img.transpose(Image.FLIP_LEFT_RIGHT)
    if random.random() < 0.4:
        img = ImageEnhance.Brightness(img).enhance(random.uniform(0.7, 1.3))
    if random.random() < 0.4:
        img = ImageEnhance.Contrast(img).enhance(random.uniform(0.7, 1.3))
    return img

for cls in classes:
    input_class = os.path.join(INPUT_DIR, cls)
    output_class = os.path.join(OUTPUT_DIR, cls)
    os.makedirs(output_class, exist_ok=True)

    files = [f for f in os.listdir(input_class) if os.path.isfile(os.path.join(input_class, f))]
    print(f"{cls}: {len(files)} Trainingsbilder gefunden")

    for file in files:
        img_path = os.path.join(input_class, file)

        with Image.open(img_path) as img:
            img = img.convert("RGB")
            base_img = img.copy()

        base_img.save(os.path.join(output_class, file))

        for i in range(2):
            aug = augment_image(base_img.copy())
            base, ext = os.path.splitext(file)
            aug.save(os.path.join(output_class, f"{base}_aug{i}{ext}"))

print("Augmentation fertig!")


In [ ]:
for cls in classes:
    count = len(os.listdir(os.path.join("dataset-augmented/train", cls)))
    print(cls, ":", count)


## 10. Training – 6 Klassen mit vorhandenen Daten

Dieses Kapitel trainiert ein CNN direkt auf den bereits vorhandenen Datenordnern und erzeugt **keine neuen Datensatzordner**.

### Verwendete Klassen
- `biological`
- `cardboard`
- `glass`
- `metal`
- `paper`
- `plastic`

### Wichtig
- `trash` wird nicht mehr verwendet
- Es werden nur die gewünschten Klassen geladen
- Für das Training kann optional der augmentierte Trainingsordner genutzt werden


In [ ]:
TARGET_CLASSES = ["biological", "cardboard", "glass", "metal", "paper", "plastic"]
USE_AUGMENTED_TRAIN = True

if USE_AUGMENTED_TRAIN:
    train_dir = os.path.join("dataset-augmented", "train")
else:
    train_dir = os.path.join("dataset-split", "train")

val_dir = os.path.join("dataset-split", "val")
test_dir = os.path.join("dataset-split", "test")

IMG_SIZE = (224, 224)
BATCH_SIZE = 32
SEED = 42
EPOCHS = 20

print("Train-Verzeichnis:", train_dir)
print("Validation-Verzeichnis:", val_dir)
print("Test-Verzeichnis:", test_dir)
print("Zielklassen:", TARGET_CLASSES)


## 11. Vorhandene Ordner prüfen

Hier wird geprüft, ob die Verzeichnisse existieren und welche Klassenordner darin liegen.


In [ ]:
print("train_dir existiert:", os.path.isdir(train_dir))
print("val_dir existiert:", os.path.isdir(val_dir))
print("test_dir existiert:", os.path.isdir(test_dir))

print("\nKlassen in train_dir:", os.listdir(train_dir) if os.path.isdir(train_dir) else "fehlt")
print("Klassen in val_dir:", os.listdir(val_dir) if os.path.isdir(val_dir) else "fehlt")
print("Klassen in test_dir:", os.listdir(test_dir) if os.path.isdir(test_dir) else "fehlt")

missing_train = [cls for cls in TARGET_CLASSES if not os.path.isdir(os.path.join(train_dir, cls))]
missing_val = [cls for cls in TARGET_CLASSES if not os.path.isdir(os.path.join(val_dir, cls))]
missing_test = [cls for cls in TARGET_CLASSES if not os.path.isdir(os.path.join(test_dir, cls))]

if missing_train or missing_val or missing_test:
    raise FileNotFoundError(
        "Es fehlen Klassenordner.\n"
        f"Fehlend in train: {missing_train}\n"
        f"Fehlend in val: {missing_val}\n"
        f"Fehlend in test: {missing_test}\n"
        "Prüfe, ob die Datenaufbereitung vorher vollständig gelaufen ist."
    )


## 12. Dateipfade nur für die gewünschten Klassen sammeln

Da in den Ordnern andere Klassen liegen können, sammeln wir gezielt nur die Klassen aus `TARGET_CLASSES`.


In [ ]:
label_to_index = {label: idx for idx, label in enumerate(TARGET_CLASSES)}
index_to_label = {idx: label for label, idx in label_to_index.items()}

def collect_filepaths(base_dir, class_names):
    filepaths = []
    labels = []

    for cls in class_names:
        cls_dir = os.path.join(base_dir, cls)

        if not os.path.isdir(cls_dir):
            raise FileNotFoundError(f"Klassenordner nicht gefunden: {cls_dir}")

        for file in os.listdir(cls_dir):
            path = os.path.join(cls_dir, file)
            if os.path.isfile(path):
                filepaths.append(path)
                labels.append(label_to_index[cls])

    return pd.DataFrame({
        "filepath": filepaths,
        "label": labels
    })

train_df = collect_filepaths(train_dir, TARGET_CLASSES)
val_df = collect_filepaths(val_dir, TARGET_CLASSES)
test_df = collect_filepaths(test_dir, TARGET_CLASSES)

print("Train:", len(train_df), "Bilder")
print("Val:", len(val_df), "Bilder")
print("Test:", len(test_df), "Bilder")

train_df.head()


## 13. Klassenverteilung prüfen

Diese Zelle zeigt, wie viele Bilder pro Klasse in Train, Validation und Test vorhanden sind.


In [ ]:
def class_counts(df, name):
    counts = df["label"].map(index_to_label).value_counts().sort_index()
    print(f"\n{name}")
    print(counts)

class_counts(train_df, "Train")
class_counts(val_df, "Validation")
class_counts(test_df, "Test")


## 14. TensorFlow-Datasets bauen

Hier werden die Dateipfade in TensorFlow-Datasets umgewandelt.  
Die Bilder werden geladen, auf `224x224` gebracht und als Tensor vorbereitet.


In [ ]:
def load_and_preprocess_image(path, label):
    image_data = tf.io.read_file(path)
    image_data = tf.image.decode_image(image_data, channels=3, expand_animations=False)
    image_data = tf.image.resize(image_data, IMG_SIZE)
    image_data = tf.cast(image_data, tf.float32)
    return image_data, label

def make_dataset(df, training=False):
    paths = df["filepath"].values
    labels = df["label"].values.astype(np.int32)

    ds = tf.data.Dataset.from_tensor_slices((paths, labels))

    if training:
        ds = ds.shuffle(buffer_size=len(df), seed=SEED)

    ds = ds.map(load_and_preprocess_image, num_parallel_calls=tf.data.AUTOTUNE)
    ds = ds.batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)
    return ds

train_ds = make_dataset(train_df, training=True)
val_ds = make_dataset(val_df, training=False)
test_ds = make_dataset(test_df, training=False)

print("Datasets erstellt.")


## 15. Beispielbilder aus dem Trainingsdatensatz anzeigen

Hier werden einige Trainingsbilder angezeigt, um zu prüfen, ob die Daten korrekt geladen wurden.


In [ ]:
plt.figure(figsize=(10, 10))

for images, labels in train_ds.take(1):
    for i in range(min(9, len(images))):
        ax = plt.subplot(3, 3, i + 1)
        plt.imshow(tf.cast(images[i], tf.uint8).numpy())
        plt.title(index_to_label[int(labels[i].numpy())])
        plt.axis("off")

plt.tight_layout()
plt.show()


## 16. CNN-Modell definieren

Hier wird das neuronale Netzwerk aufgebaut.


In [ ]:
model = Sequential([
    layers.Input(shape=(224, 224, 3)),
    layers.Rescaling(1. / 255),

    layers.Conv2D(32, (3, 3), activation="relu"),
    layers.MaxPooling2D(),

    layers.Conv2D(64, (3, 3), activation="relu"),
    layers.MaxPooling2D(),

    layers.Conv2D(128, (3, 3), activation="relu"),
    layers.MaxPooling2D(),

    layers.Flatten(),
    layers.Dense(128, activation="relu"),
    layers.Dropout(0.3),
    layers.Dense(len(TARGET_CLASSES), activation="softmax")
])


## 17. Modell kompilieren

In [ ]:
model.compile(
    optimizer="adam",
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

model.summary()


## 18. Modell trainieren

In [ ]:
early_stopping = keras.callbacks.EarlyStopping(
    monitor="val_loss",
    patience=5,
    restore_best_weights=True
)

history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=EPOCHS,
    callbacks=[early_stopping]
)


## 19. Accuracy und Loss visualisieren

In [ ]:
acc = history.history["accuracy"]
val_acc = history.history["val_accuracy"]
loss = history.history["loss"]
val_loss = history.history["val_loss"]

epochs_range = range(1, len(acc) + 1)

plt.figure(figsize=(12, 5))

plt.subplot(1, 2, 1)
plt.plot(epochs_range, acc, label="Training Accuracy")
plt.plot(epochs_range, val_acc, label="Validation Accuracy")
plt.legend()
plt.title("Accuracy")

plt.subplot(1, 2, 2)
plt.plot(epochs_range, loss, label="Training Loss")
plt.plot(epochs_range, val_loss, label="Validation Loss")
plt.legend()
plt.title("Loss")

plt.tight_layout()
plt.show()


## 20. Testdaten auswerten

In [ ]:
test_loss, test_acc = model.evaluate(test_ds)

print(f"Test Loss: {test_loss:.4f}")
print(f"Test Accuracy: {test_acc:.4f}")


## 21. Classification Report und Confusion Matrix

In [ ]:
y_true = []
y_pred = []

for images_batch, labels_batch in test_ds:
    preds = model.predict(images_batch, verbose=0)
    preds = np.argmax(preds, axis=1)

    y_true.extend(labels_batch.numpy())
    y_pred.extend(preds)

print(classification_report(y_true, y_pred, target_names=TARGET_CLASSES))

cm = confusion_matrix(y_true, y_pred)
cm_df = pd.DataFrame(cm, index=TARGET_CLASSES, columns=TARGET_CLASSES)

print(cm_df)

plt.figure(figsize=(8, 7))
plt.imshow(cm, interpolation="nearest")
plt.title("Confusion Matrix")
plt.colorbar()

tick_marks = np.arange(len(TARGET_CLASSES))
plt.xticks(tick_marks, TARGET_CLASSES, rotation=45)
plt.yticks(tick_marks, TARGET_CLASSES)

for i in range(len(TARGET_CLASSES)):
    for j in range(len(TARGET_CLASSES)):
        plt.text(j, i, cm[i, j], ha="center", va="center")

plt.ylabel("Tatsächliche Klasse")
plt.xlabel("Vorhergesagte Klasse")
plt.tight_layout()
plt.show()


## 22. Zufällige Beispielvorhersagen

In [ ]:
sample_df = test_df.sample(min(9, len(test_df)), random_state=None)

plt.figure(figsize=(10, 10))

for i, row in enumerate(sample_df.itertuples()):
    img = tf.io.read_file(row.filepath)
    img = tf.image.decode_image(img, channels=3, expand_animations=False)
    img = tf.image.resize(img, IMG_SIZE)
    img_input = tf.expand_dims(img, axis=0)

    pred = model.predict(img_input, verbose=0)
    pred_class = TARGET_CLASSES[int(np.argmax(pred[0]))]
    true_class = TARGET_CLASSES[int(row.label)]

    ax = plt.subplot(3, 3, i + 1)
    plt.imshow(tf.cast(img, tf.uint8).numpy())
    plt.title(f"True: {true_class}\nPred: {pred_class}")
    plt.axis("off")

plt.tight_layout()
plt.show()


## 23. Modell speichern

Das trainierte Modell wird als `.keras`-Datei gespeichert und kann später wieder geladen werden.  
Diese Datei sollte nicht ins Git-Repo gepusht werden.


In [ ]:
MODEL_PATH = "trash_classifier_6classes.keras"
model.save(MODEL_PATH)
print("Modell gespeichert unter:", MODEL_PATH)


## 24. Predictor Notebook – Müllerkennung

Dieses Kapitel ist getrennt vom Training und dient nur dazu, mit dem bereits trainierten Modell neue Bilder vorherzusagen.

### Ziel
Du lädst ein beliebiges Bild in den Projektordner oder gibst einen Bildpfad an, und das Modell sagt vorher, zu welcher der folgenden Klassen das Bild gehört:

- `biological`
- `cardboard`
- `glass`
- `metal`
- `paper`
- `plastic`


In [ ]:
REQUIRED_FILE = "trash_classifier_6classes.keras"

def find_project_root_for_model(start_path, required_file=REQUIRED_FILE):
    current = start_path
    while True:
        if required_file in os.listdir(current):
            return current
        parent = os.path.dirname(current)
        if parent == current:
            return None
        current = parent

project_root_for_model = find_project_root_for_model(os.getcwd())

if project_root_for_model is None:
    raise FileNotFoundError(f"'{REQUIRED_FILE}' konnte nicht gefunden werden.")

if os.getcwd() != project_root_for_model:
    os.chdir(project_root_for_model)
    print("Arbeitsverzeichnis gesetzt auf:", project_root_for_model)
else:
    print("Arbeitsverzeichnis war schon korrekt.")


## 25. Modell laden

In [ ]:
MODEL_PATH = "trash_classifier_6classes.keras"
model = keras.models.load_model(MODEL_PATH)

print("Modell erfolgreich geladen.")
model.summary()


## 26. Klassen und Bildgröße festlegen

Diese Werte müssen zur Training-Datei passen.


In [ ]:
TARGET_CLASSES = ["biological", "cardboard", "glass", "metal", "paper", "plastic"]
IMG_SIZE = (224, 224)

print("Klassen:", TARGET_CLASSES)
print("Bildgröße:", IMG_SIZE)


## 27. Vorhersagefunktion für ein einzelnes Bild

Diese Funktion:
- lädt ein Bild
- passt es auf `224x224` an
- gibt die vorhergesagte Klasse aus
- zeigt zusätzlich die Wahrscheinlichkeiten pro Klasse


In [ ]:
def predict_image(img_path):
    if not os.path.exists(img_path):
        raise FileNotFoundError(f"Bild nicht gefunden: {img_path}")

    img = image.load_img(img_path, target_size=IMG_SIZE)
    img_array = image.img_to_array(img)
    img_array = np.expand_dims(img_array, axis=0)

    preds = model.predict(img_array, verbose=0)[0]
    pred_index = int(np.argmax(preds))
    pred_class = TARGET_CLASSES[pred_index]

    plt.figure(figsize=(5, 5))
    plt.imshow(img)
    plt.title(f"Prediction: {pred_class}")
    plt.axis("off")
    plt.show()

    print("Vorhergesagte Klasse:", pred_class)
    print("\nWahrscheinlichkeiten:")
    for class_name, prob in zip(TARGET_CLASSES, preds):
        print(f"  {class_name}: {prob:.4f}")

    return pred_class, preds


## 28. Ein einzelnes Bild testen

Lege ein Bild in deinen Projektordner oder in einen Unterordner und passe den Pfad unten an.

Beispiele:
- `testbild.jpg`
- `bilder/testbild.jpg`
- `src/predict_images/gest.jpg`


In [ ]:
img_path = r"src\predict_images\gest.jpg"

# Beispielaufruf:
# predict_image(img_path)


## 29. Mehrere Bilder nacheinander testen

In [ ]:
def predict_images_from_folder(folder_path):
    if not os.path.isdir(folder_path):
        raise FileNotFoundError(f"Ordner nicht gefunden: {folder_path}")

    valid_extensions = (".jpg", ".jpeg", ".png", ".bmp", ".webp")
    files = [f for f in os.listdir(folder_path) if f.lower().endswith(valid_extensions)]

    if not files:
        print("Keine unterstützten Bilddateien gefunden.")
        return

    print(f"{len(files)} Bild(er) gefunden in: {folder_path}\n")

    for file in files:
        full_path = os.path.join(folder_path, file)
        print("=" * 60)
        print("Datei:", full_path)
        predict_image(full_path)

# Beispiel:
# predict_images_from_folder("predict_images")


## 30. Optional: Ein Testbild aus dem vorhandenen Datensatz prüfen

In [ ]:
example_path = os.path.join("dataset-split", "test", "glass")

if os.path.isdir(example_path):
    files = os.listdir(example_path)
    if files:
        sample_image = os.path.join(example_path, files[0])
        print("Beispielbild:", sample_image)
    else:
        print("Keine Dateien im Beispielordner gefunden.")
else:
    print("Beispielordner nicht gefunden.")

# Optional:
# predict_image(sample_image)


## Fazit

Das Projekt zeigt, wie wichtig eine strukturierte und transparente Datenvorbereitung für den Erfolg eines Machine-Learning-Modells ist. Durch die Vereinheitlichung der Bildgrößen, die gezielte Aufteilung in Trainings-, Validierungs- und Testmengen sowie die datenbasierte Erweiterung mittels Augmentation konnte ein belastbarer und vielseitiger Datensatz geschaffen werden.

Darüber hinaus verdeutlicht das Projekt, dass Data Preparation weit mehr ist als ein technischer Zwischenschritt: Sie entscheidet maßgeblich über die Modellqualität, die Trainingsstabilität und die spätere Anwendbarkeit der KI.
